# 🎨 AbiArt — Kaggle GPU Backend (Automatic1111 SD WebUI API)

### ⚡ Configuration requise dans Kaggle (panneau à droite)
1. **Accelerator** → `GPU T4 x2` (ou `P100`)
2. **Internet** → `ON`
3. Cliquez sur **Run All** (ou exécutez les cellules 1 à 5 dans l'ordre).
4. À la fin, copiez l'URL Cloudflare affichée (`https://xxxx.trycloudflare.com`) et collez-la dans AbiArt → Backend Kaggle → Connect.

### 🔧 Pourquoi cette version ?
Kaggle tourne maintenant sous **Python 3.13** avec **transformers 5.x**. Automatic1111 (v1.10.1) a été écrit pour
**Python 3.10 + torch 2.1.2 + transformers 4.30.2** : le chargement du modèle plantait avec
`RuntimeError: cannot schedule new futures after interpreter shutdown`.
Ce notebook crée donc un **environnement Python 3.10 isolé** (via `uv`) uniquement pour A1111, sans toucher à l'environnement Kaggle.

📥 La cellule 5 installe aussi une petite extension qui permet au site AbiArt de télécharger des modèles Civitai / Hugging Face directement sur Kaggle.

⏱️ Première exécution : ~10 à 15 min (installation + téléchargement du modèle de 6,9 Go). Ensuite, la cellule 5 démarre en 1-2 min.

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 1 — VÉRIFICATION DU GPU
# ══════════════════════════════════════════════════
import subprocess, shutil

has_gpu = False
if shutil.which('nvidia-smi'):
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                           capture_output=True, text=True)
        if r.returncode == 0 and r.stdout.strip():
            has_gpu = True
            print('✓ GPU actif et détecté :')
            print('  ' + r.stdout.strip().replace('\n', '\n  '))
    except Exception:
        pass

if not has_gpu:
    print('❌ ERREUR CRITIQUE : le GPU n\'est pas activé sur Kaggle !')
    print('    → Volet de droite → "Session options" → "Accelerator"')
    print('    → Choisissez "GPU T4 x2" (ou "GPU P100")')
    print('    → Vérifiez que "Internet" est sur "On"')
    print('    → Redémarrez la session.')
    raise RuntimeError('Veuillez activer le GPU dans le panneau de droite avant de continuer.')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 2 — INSTALLATION (Python 3.10 isolé + A1111 v1.10.1)
# ══════════════════════════════════════════════════
import os, sys, shutil, subprocess

def sh(cmd, cwd=None, env=None):
    """Exécute une commande shell et affiche la sortie en direct dans la cellule."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'Commande échouée (code {p.returncode}) : {cmd}')

# 0. Choix du dossier de travail (celui qui a le plus d'espace libre ; /kaggle/working est limité à 20 Go)
cands = [d for d in ('/kaggle/temp', '/tmp') if os.path.isdir(d) and os.access(d, os.W_OK)] or ['/kaggle/working']
BASE = max(cands, key=lambda d: shutil.disk_usage(d).free) + '/abiart'
os.makedirs(BASE, exist_ok=True)
with open('/tmp/abiart_base.txt', 'w') as f:
    f.write(BASE)

WEBUI = f'{BASE}/stable-diffusion-webui'
VENV  = f'{BASE}/venv'
PY    = f'{VENV}/bin/python'
UV    = f'"{sys.executable}" -m uv'
print(f'Dossier de travail : {BASE}  ({shutil.disk_usage(BASE).free / 1024**3:.1f} Go libres)')

# 1. uv (installeur rapide, capable de télécharger Python 3.10)
print('\n[1/5] Installation de uv...')
sh(f'"{sys.executable}" -m pip install -q -U uv')

# 2. Environnement virtuel Python 3.10
print('\n[2/5] Création de l\'environnement Python 3.10...')
if not os.path.exists(PY):
    sh(f'{UV} venv --seed --python 3.10 "{VENV}"')
sh(f'"{PY}" --version')

# 3. Cloner Automatic1111 (version stable v1.10.1)
print('\n[3/5] Clonage d\'Automatic1111 v1.10.1...')
if not os.path.exists(f'{WEBUI}/launch.py'):
    shutil.rmtree(WEBUI, ignore_errors=True)
    sh(f'git clone -q --depth 1 --branch v1.10.1 https://github.com/AUTOMATIC1111/stable-diffusion-webui.git "{WEBUI}"')
else:
    print('  ✓ déjà présent')

# 4. Dépendances : exactement celles qu'A1111 attend (requirements_versions.txt) + torch 2.1.2
print('\n[4/5] Installation des dépendances Python (torch 2.1.2, transformers 4.30.2, gradio 3.41.2...)')
marker = f'{VENV}/.abiart_deps_ok'
if not os.path.exists(marker):
    lines = []
    for l in open(f'{WEBUI}/requirements_versions.txt').read().splitlines():
        l = l.strip()
        if l and l != 'torch':
            lines.append(l)
    lines += ['torch==2.1.2', 'torchvision==0.16.2', 'ftfy', 'regex']
    req = f'{BASE}/requirements_kaggle.txt'
    with open(req, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    sh(f'{UV} pip install --python "{PY}" -r "{req}"')
    # CLIP : doit être compilé avec le setuptools épinglé (69.5.1) qui contient encore pkg_resources
    sh(f'{UV} pip install --python "{PY}" --no-build-isolation --no-deps '
       f'https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip')
    open(marker, 'w').write('ok')
else:
    print('  ✓ déjà installées')

# 5. Sous-dépôts d'A1111 (stable-diffusion, generative-models, k-diffusion, BLIP, assets)
print('\n[5/5] Clonage des sous-dépôts A1111...')
env = os.environ.copy()
for k in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    env.pop(k, None)
env['STABLE_DIFFUSION_REPO'] = 'https://github.com/w-e-w/stablediffusion.git'  # dépôt Stability-AI d'origine retiré
env['PATH'] = f'{VENV}/bin:' + env.get('PATH', '')
env['MPLBACKEND'] = 'Agg'
sh(f'"{PY}" launch.py --exit --skip-torch-cuda-test --skip-install', cwd=WEBUI, env=env)

total, used, free = shutil.disk_usage(BASE)
print(f'\n✓ Environnement prêt ! Espace libre : {free / 1024**3:.1f} Go')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 3 — VÉRIFICATION DE L'ENVIRONNEMENT PYTHON 3.10
# ══════════════════════════════════════════════════
import os, subprocess

BASE  = open('/tmp/abiart_base.txt').read().strip()
WEBUI = f'{BASE}/stable-diffusion-webui'
PY    = f'{BASE}/venv/bin/python'

# Environnement propre : pas de PYTHONPATH Kaggle, pas de bibliothèques CUDA système
# (elles entreraient en conflit avec celles embarquées dans torch 2.1.2)
env = os.environ.copy()
for k in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    env.pop(k, None)
env['LD_LIBRARY_PATH'] = ':'.join(p for p in env.get('LD_LIBRARY_PATH', '').split(':') if p and 'cuda' not in p.lower())
env['CUDA_VISIBLE_DEVICES'] = '0'
env['MPLBACKEND'] = 'Agg'  # Kaggle force un backend matplotlib Jupyter absent du venv 3.10

test = r"""
import sys, importlib
print('Python        :', sys.version.split()[0])
import torch
print('torch         :', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU dispo     :', torch.cuda.is_available())
assert torch.cuda.is_available(), "torch ne voit pas le GPU"
print('GPU           :', torch.cuda.get_device_name(0))
x = torch.randn(1024, 1024, device='cuda', dtype=torch.float16)
print('Test calcul   :', 'OK' if torch.isfinite((x @ x).sum()).item() else 'NaN')
bad = 0
for m in ['transformers', 'gradio', 'fastapi', 'pytorch_lightning', 'open_clip', 'clip', 'safetensors',
          'accelerate', 'omegaconf', 'kornia', 'einops', 'torchsde', 'spandrel', 'diskcache', 'skimage']:
    try:
        mod = importlib.import_module(m)
        print(f'  ✓ {m:<18}', getattr(mod, '__version__', ''))
    except Exception as e:
        bad += 1
        print(f'  ❌ {m:<18}', e)
sys.exit(1 if bad else 0)
"""
r = subprocess.run([PY, '-c', test], env=env, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError('Environnement incomplet : relancez la cellule 2, puis celle-ci.')
print('✓ Environnement Python 3.10 opérationnel.')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 4 — TÉLÉCHARGEMENT DES MODÈLES (Animagine XL 3.1 + SDXL VAE fp16-fix)
# ══════════════════════════════════════════════════
import os, shutil, subprocess

BASE  = open('/tmp/abiart_base.txt').read().strip()
WEBUI = f'{BASE}/stable-diffusion-webui'
ckpt_dir = f'{WEBUI}/models/Stable-diffusion'
vae_dir  = f'{WEBUI}/models/VAE'
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

def download(url, dest, min_bytes, label):
    if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
        print(f'✓ {label} déjà présent')
        return
    print(f'Téléchargement de {label}...')
    for attempt in range(1, 4):
        r = subprocess.run(['wget', '-c', '-q', '--show-progress', '--progress=dot:giga',
                            '--tries=3', '--timeout=60', url, '-O', dest])
        if r.returncode == 0 and os.path.getsize(dest) >= min_bytes:
            print(f'✓ {label} OK ({os.path.getsize(dest) / 1024**3:.2f} Go)')
            return
        print(f'  ⚠️ tentative {attempt}/3 échouée, nouvel essai...')
    raise RuntimeError(f'Téléchargement de {label} impossible (Internet est-il sur ON ?)')

download('https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors',
         f'{ckpt_dir}/animagine-xl-3.1.safetensors', 6_000_000_000, 'Animagine XL 3.1 (~6,9 Go)')

# VAE "fp16-fix" : le VAE SDXL d'origine produit des images noires/NaN en demi-précision sur T4/P100
download('https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl.vae.safetensors',
         f'{vae_dir}/sdxl_vae_fp16_fix.safetensors', 300_000_000, 'SDXL VAE fp16-fix (~335 Mo)')

total, used, free = shutil.disk_usage(BASE)
print(f'\n✓ Modèles prêts ! Espace libre : {free / 1024**3:.1f} Go')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 5 — LANCEMENT DE L'API A1111 + TUNNEL CLOUDFLARE
# (le tunnel n'est ouvert qu'une fois le modèle réellement chargé)
# ══════════════════════════════════════════════════
import os, re, sys, time, shutil, signal, subprocess, requests

BASE  = open('/tmp/abiart_base.txt').read().strip()
WEBUI = f'{BASE}/stable-diffusion-webui'
PY    = f'{BASE}/venv/bin/python'
PORT  = 7860
LOG   = '/tmp/webui.log'
TLOG  = '/tmp/tunnel.log'
CF    = '/tmp/cloudflared'

# Optionnel : URL de votre site AbiArt déployé, pour l'auto-connexion
# Ex: ABIART_APP_URL = "https://mon-vrai-site.up.railway.app"
ABIART_APP_URL = ""

ckpt = f'{WEBUI}/models/Stable-diffusion/animagine-xl-3.1.safetensors'
vae  = f'{WEBUI}/models/VAE/sdxl_vae_fp16_fix.safetensors'
if not os.path.exists(ckpt):
    raise RuntimeError('Modèle absent : exécutez la cellule 4.')

# 0. Nettoyage d'une éventuelle exécution précédente
subprocess.run(['pkill', '-9', '-f', f'{WEBUI}/launch.py'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 1. Environnement propre (voir cellule 3)
env = os.environ.copy()
for k in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    env.pop(k, None)
env['LD_LIBRARY_PATH'] = ':'.join(p for p in env.get('LD_LIBRARY_PATH', '').split(':') if p and 'cuda' not in p.lower())
env['PATH'] = f'{BASE}/venv/bin:' + env.get('PATH', '')
env['CUDA_VISIBLE_DEVICES'] = '0'
env['MPLBACKEND'] = 'Agg'  # Kaggle force un backend matplotlib Jupyter absent du venv 3.10
env['PYTHONUNBUFFERED'] = '1'
env['STABLE_DIFFUSION_REPO'] = 'https://github.com/w-e-w/stablediffusion.git'
env['ABIART_LOCAL_PORT'] = str(PORT)  # lu par l'extension abiart-downloader (generation asynchrone)

# 1b. Extension AbiArt : permet au site de télécharger des modèles Civitai / Hugging Face sur cette machine
EXT_DIR = f'{WEBUI}/extensions/abiart-downloader/scripts'
os.makedirs(EXT_DIR, exist_ok=True)
EXT_SRC = r"""# AbiArt - telechargement de modeles (Civitai / Hugging Face) vers A1111 depuis le site.
#
# Ajoute 3 routes a l'API d'Automatic1111 (meme port, donc meme tunnel Cloudflare) :
#   GET  /abiart/ping                 -> detecte la presence de l'extension
#   POST /abiart/download             -> demarre un telechargement en arriere-plan, renvoie un job_id
#   GET  /abiart/download/{job_id}    -> progression (running / done / error)
#
# Securite : https uniquement, hotes de depart limites (civitai.com, civitai.red, huggingface.co),
# redirections suivies a la main (adresses privees refusees), extensions de fichier et taille limitees.
import ipaddress
import os
import re
import shutil
import socket
import threading
import time
import uuid
from urllib.parse import urljoin, urlparse

import requests

ALLOWED_HOSTS = ("civitai.com", "civitai.red", "huggingface.co")
ALLOWED_EXT = (".safetensors", ".ckpt", ".pt", ".pth")
MAX_BYTES = 14 * 1024 ** 3
MAX_ACTIVE_JOBS = 2
SUBDIRS = {"checkpoint": "Stable-diffusion", "lora": "Lora", "vae": "VAE"}

JOBS = {}
_LOCK = threading.Lock()


def _models_path():
    try:
        from modules import paths_internal
        return paths_internal.models_path
    except Exception:
        return os.environ.get("ABIART_MODELS_PATH", "models")


def _host_allowed(host):
    host = (host or "").lower()
    return any(host == d or host.endswith("." + d) for d in ALLOWED_HOSTS)


def _is_public(host):
    try:
        infos = socket.getaddrinfo(host, 443, proto=socket.IPPROTO_TCP)
    except Exception:
        return False
    return bool(infos) and all(ipaddress.ip_address(i[4][0]).is_global for i in infos)


def _safe_filename(name):
    name = os.path.basename((name or "").replace("\\", "/")).strip()
    name = re.sub(r"[^A-Za-z0-9._\-\(\) \[\]]", "_", name).lstrip(".")[:150]
    if not name.lower().endswith(ALLOWED_EXT):
        raise ValueError("Unsupported file type (allowed: " + ", ".join(ALLOWED_EXT) + ")")
    return name


def _clean_name(name):
    # Comme _safe_filename, mais renvoie "" (au lieu de lever) si le nom n'est pas un fichier de modele.
    try:
        return _safe_filename(name)
    except ValueError:
        return ""


def _open(url, token=""):
    # GET en streaming ; les redirections sont suivies a la main pour verifier chaque saut.
    current = url
    for hop in range(6):
        p = urlparse(current)
        host = p.hostname or ""
        if p.scheme != "https":
            raise ValueError("Only https URLs are allowed")
        if hop == 0 and not _host_allowed(host):
            raise ValueError("Host not allowed: " + host)
        if not _is_public(host):
            raise ValueError("Cannot reach %s from this machine, or its address is not public" % host)
        headers = {"User-Agent": "AbiArt-Kaggle/1.0"}
        if hop == 0 and token and "civitai" in host:
            headers["Authorization"] = "Bearer " + token  # jamais transmis apres une redirection
        r = requests.get(current, headers=headers, stream=True, allow_redirects=False, timeout=(15, 60))
        if r.status_code in (301, 302, 303, 307, 308) and r.headers.get("Location"):
            current = urljoin(current, r.headers["Location"])
            r.close()
            continue
        return r
    raise ValueError("Too many redirects")


def _header_filename(resp):
    cd = resp.headers.get("Content-Disposition", "")
    m = re.search(r"filename\*?=(?:UTF-8'')?\"?([^\";]+)\"?", cd, re.I)
    return m.group(1) if m else ""


def _worker(job_id, url, token, kind, wanted_name):
    job = JOBS[job_id]
    tmp = None
    try:
        r = _open(url, token)
        if r.status_code in (401, 403):
            raise RuntimeError("Download refused (HTTP %d): this model probably needs a Civitai API token." % r.status_code)
        if r.status_code != 200:
            raise RuntimeError("Download failed (HTTP %d)" % r.status_code)
        if "text/html" in r.headers.get("Content-Type", ""):
            raise RuntimeError("The link returned a web page, not a model file (login required?)")
        header_name = _header_filename(r)
        name = _clean_name(wanted_name) or _clean_name(header_name)
        if not name:
            if header_name:  # Civitai envoie un fichier qui n'est pas un modele (zip, yaml, ...)
                raise RuntimeError('Ce lien ne pointe pas vers un modele : fichier recu "%s" (attendu : %s)'
                                   % (header_name, ", ".join(ALLOWED_EXT)))
            name = "model_%s.safetensors" % job_id[:6]
        total = int(r.headers.get("Content-Length") or 0)
        if total > MAX_BYTES:
            raise RuntimeError("File too large")
        dest_dir = os.path.join(_models_path(), SUBDIRS[kind])
        os.makedirs(dest_dir, exist_ok=True)
        if total and shutil.disk_usage(dest_dir).free < total + 1024 ** 3:
            raise RuntimeError("Not enough free disk space on the Kaggle machine")
        final = os.path.join(dest_dir, name)
        job.update(filename=name, total=total)
        if os.path.exists(final) and os.path.getsize(final) > 0:
            size = os.path.getsize(final)
            job.update(state="done", done=size, total=size, note="already present")
            return
        tmp = final + ".part"
        done = 0
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if done > MAX_BYTES:
                    raise RuntimeError("File too large")
                job["done"] = done
        if total and done != total:
            raise RuntimeError("Incomplete download")
        os.replace(tmp, final)
        tmp = None
        job.update(state="done", done=done, total=done)
    except Exception as e:
        job.update(state="error", error=str(e))
    finally:
        if tmp and os.path.exists(tmp):
            try:
                os.remove(tmp)
            except OSError:
                pass


def start_download(data):
    # Retourne (payload, http_status).
    url = (data.get("url") or "").strip()
    kind = (data.get("type") or "checkpoint").lower()
    if kind not in SUBDIRS:
        return {"error": "Unknown type: " + kind}, 400
    if not url:
        return {"error": "Missing url"}, 400
    # Nom suggere par le site : garde s'il est valide, sinon ignore (le vrai nom est lu dans la reponse de Civitai).
    wanted = _clean_name((data.get("filename") or "").strip())
    host = urlparse(url).hostname or ""
    if urlparse(url).scheme != "https" or not _host_allowed(host):
        return {"error": "Only https links from civitai.com, civitai.red or huggingface.co are allowed"}, 400
    with _LOCK:
        if sum(1 for j in JOBS.values() if j["state"] == "running") >= MAX_ACTIVE_JOBS:
            return {"error": "Too many downloads in progress, wait for one to finish"}, 429
        job_id = uuid.uuid4().hex[:12]
        JOBS[job_id] = {"state": "running", "kind": kind, "filename": wanted, "done": 0, "total": 0,
                        "error": "", "started": time.time()}
    threading.Thread(target=_worker, args=(job_id, url, (data.get("token") or "").strip(), kind, wanted),
                     daemon=True).start()
    return {"success": True, "job_id": job_id, "filename": wanted, "type": kind}, 200


def job_status(job_id):
    job = JOBS.get(job_id)
    if not job:
        return {"error": "Unknown job (the notebook may have been restarted)", "state": "error"}, 404
    return {k: job.get(k) for k in ("state", "kind", "filename", "done", "total", "error", "note")}, 200


# ---------------------------------------------------------------------------------------------
# Generation asynchrone : evite la coupure a ~100 s du tunnel Cloudflare (erreur HTTP 524).
# Le site demarre la generation (reponse immediate : un job_id) puis demande l'etat toutes les 2 s.
# Aucune requete HTTP ne dure plus de quelques secondes, quelle que soit la duree du calcul.
# ---------------------------------------------------------------------------------------------
LOCAL_API = "http://127.0.0.1:%s" % os.environ.get("ABIART_LOCAL_PORT", "7860")
GEN_TTL = 15 * 60
GEN_MAX_ACTIVE = 4
GEN_JOBS = {}
_GEN_RUN = threading.Lock()   # A1111 calcule une image a la fois : les suivantes attendent leur tour
_GEN_LOCK = threading.Lock()


def _gen_error(resp):
    try:
        js = resp.json()
        msg = js.get("error") or js.get("detail") or js.get("errors") or str(js)
    except Exception:
        msg = resp.text or ("HTTP %d" % resp.status_code)
    return str(msg)[:400]


def _gen_worker(job_id, payload):
    job = GEN_JOBS[job_id]
    with _GEN_RUN:
        if job.get("cancel"):
            job.update(state="cancelled", error="Cancelled", updated=time.time())
            return
        job.update(state="running", started=time.time(), updated=time.time())
        try:
            r = requests.post(LOCAL_API + "/sdapi/v1/txt2img", json=payload, timeout=3600)
            if job.get("cancel"):
                job.update(state="cancelled", error="Cancelled", updated=time.time())
            elif r.status_code == 200:
                data = r.json()
                job.update(state="done", images=data.get("images", []), info=data.get("info"), updated=time.time())
            else:
                job.update(state="error", error=_gen_error(r), updated=time.time())
        except Exception as e:
            job.update(state="error", error=("%s: %s" % (type(e).__name__, e))[:400], updated=time.time())


def start_generate(data):
    # Retourne (payload, http_status). `data` = corps d'un appel /sdapi/v1/txt2img.
    if not isinstance(data, dict) or not data.get("prompt"):
        return {"error": "Missing prompt"}, 400
    now = time.time()
    with _GEN_LOCK:
        for jid in [j for j, v in GEN_JOBS.items() if v["state"] in ("done", "error", "cancelled") and now - v["updated"] > GEN_TTL]:
            GEN_JOBS.pop(jid, None)
        if sum(1 for v in GEN_JOBS.values() if v["state"] in ("queued", "running")) >= GEN_MAX_ACTIVE:
            return {"error": "Too many generations in progress, wait for one to finish"}, 429
        job_id = uuid.uuid4().hex[:16]
        GEN_JOBS[job_id] = {"state": "queued", "created": now, "updated": now, "error": "", "cancel": False}
    threading.Thread(target=_gen_worker, args=(job_id, data), daemon=True).start()
    return {"success": True, "job_id": job_id}, 200


def gen_status(job_id):
    job = GEN_JOBS.get(job_id)
    if not job:
        return {"error": "Unknown job (the notebook may have been restarted)", "state": "error"}, 404
    out = {"state": job["state"], "error": job.get("error", "")}
    if job["state"] == "queued":
        out["queue_position"] = sum(1 for v in GEN_JOBS.values()
                                    if v["state"] in ("queued", "running") and v["created"] < job["created"])
    elif job["state"] == "running":
        try:
            p = requests.get(LOCAL_API + "/sdapi/v1/progress", params={"skip_current_image": "true"}, timeout=5).json()
            out["progress"] = float(p.get("progress") or 0)
            out["eta"] = p.get("eta_relative")
        except Exception:
            out["progress"] = None
    elif job["state"] == "done":
        out["progress"] = 1.0
        out["images"] = job["images"]
        out["info"] = job["info"]
    return out, 200


def gen_cancel(job_id):
    job = GEN_JOBS.get(job_id)
    if not job:
        return {"error": "Unknown job"}, 404
    if job["state"] in ("queued", "running"):
        job["cancel"] = True
        if job["state"] == "queued":
            job.update(state="cancelled", error="Cancelled", updated=time.time())
        else:
            try:
                requests.post(LOCAL_API + "/sdapi/v1/interrupt", timeout=10)
            except Exception:
                pass
    return {"success": True}, 200


def _register(_demo, app):
    from fastapi import Request
    from fastapi.responses import JSONResponse

    @app.get("/abiart/ping")
    async def abiart_ping():
        return {"ok": True, "version": 2, "jobs": True}

    @app.post("/abiart/download")
    async def abiart_download(request: Request):
        try:
            data = await request.json()
        except Exception:
            data = {}
        payload, status = start_download(data if isinstance(data, dict) else {})
        return JSONResponse(payload, status_code=status)

    @app.get("/abiart/download/{job_id}")
    async def abiart_download_status(job_id: str):
        payload, status = job_status(job_id)
        return JSONResponse(payload, status_code=status)

    import asyncio

    async def _in_thread(fn, *args):
        return await asyncio.get_running_loop().run_in_executor(None, fn, *args)

    @app.post("/abiart/generate")
    async def abiart_generate(request: Request):
        try:
            data = await request.json()
        except Exception:
            data = {}
        payload, status = await _in_thread(start_generate, data if isinstance(data, dict) else {})
        return JSONResponse(payload, status_code=status)

    @app.get("/abiart/generate/{job_id}")
    async def abiart_generate_status(job_id: str):
        payload, status = await _in_thread(gen_status, job_id)
        return JSONResponse(payload, status_code=status)

    @app.post("/abiart/generate/{job_id}/cancel")
    async def abiart_generate_cancel(job_id: str):
        payload, status = await _in_thread(gen_cancel, job_id)
        return JSONResponse(payload, status_code=status)


try:
    from modules import script_callbacks
    script_callbacks.on_app_started(_register)
except Exception:
    pass  # import hors A1111 (tests)
"""
with open(f'{EXT_DIR}/abiart_downloader.py', 'w', encoding='utf-8') as f:
    f.write(EXT_SRC)

# 2. Lancement d'A1111 en mode API seule (processus d'arrière-plan, logs dans un fichier)
cmd = [PY, 'launch.py',
       '--nowebui', '--api', '--port', str(PORT),
       '--skip-install',
       '--opt-sdp-attention', '--medvram-sdxl', '--no-hashing',
       '--cors-allow-origins=*',
       '--ckpt', ckpt]
if os.path.exists(vae):
    cmd += ['--vae-path', vae]

logf = open(LOG, 'w')
proc = subprocess.Popen(cmd, cwd=WEBUI, env=env, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
print('Démarrage d\'Automatic1111 (chargement du modèle : 1 à 3 min)...\n')

def start_tunnel():
    if not os.path.exists(CF):
        print('Téléchargement de cloudflared...')
        subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                        '-O', CF], check=True)
        os.chmod(CF, 0o755)
    tl = open(TLOG, 'w')
    tp = subprocess.Popen([CF, 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
                          stdout=tl, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        time.sleep(1)
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', open(TLOG).read())
        if m:
            return tp, m.group(0)
    return tp, None

pos, ready, tunnel_proc, url, t0 = 0, False, None, None, time.time()
try:
    while True:
        # a) afficher les nouvelles lignes du log
        with open(LOG, 'r', errors='replace') as f:
            f.seek(pos)
            chunk = f.read()
            pos = f.tell()
        if chunk:
            print(chunk, end='')

        # b) le serveur est-il mort ?
        if proc.poll() is not None:
            print(f'\n❌ Le serveur A1111 s\'est arrêté (code {proc.returncode}). Voir les logs ci-dessus.')
            break

        if not ready:
            full = open(LOG, errors='replace').read()
            if 'stable diffusion model failed to load' in full.lower():
                print('\n❌ Le modèle Stable Diffusion n\'a pas pu se charger. Voir les logs ci-dessus.')
                proc.terminate()
                break
            if 'Model loaded in' in full:
                try:
                    ready = requests.get(f'http://127.0.0.1:{PORT}/sdapi/v1/options', timeout=5).status_code == 200
                except Exception:
                    ready = False
            if not ready and time.time() - t0 > 1500:
                print('\n❌ Délai dépassé (25 min) : le serveur n\'est jamais devenu prêt.')
                proc.terminate()
                break

            if ready:
                # c) ouvrir le tunnel Cloudflare maintenant que l'API répond
                tunnel_proc, url = start_tunnel()
                if url:
                    # laisser le DNS se propager puis tester l'URL publique
                    ok = False
                    for _ in range(15):
                        try:
                            if requests.get(url + '/sdapi/v1/sd-models', timeout=10).status_code == 200:
                                ok = True
                                break
                        except Exception:
                            pass
                        time.sleep(2)
                    print('\n' + '=' * 65)
                    print('🎨 BACKEND KAGGLE PRÊT :')
                    print(f'   >>>  {url}  <<<')
                    print('=' * 65)
                    if not ok:
                        print('⚠️ L\'URL ne répond pas encore au test, réessayez dans quelques secondes.')
                    try:
                        _ping = requests.get(f'http://127.0.0.1:{PORT}/abiart/ping', timeout=5)
                        ext_ok = _ping.status_code == 200
                        jobs_ok = ext_ok and bool(_ping.json().get('jobs'))
                    except Exception:
                        ext_ok = False
                        jobs_ok = False
                    print('\n📥 Téléchargement de modèles Civitai depuis le site : ' + ('activé ✓' if ext_ok else '⚠️ extension non chargée (voir les logs)'))
                    print('⏱️  Générations longues (Hi-Res, SDXL) sans erreur 524 : ' + ('activé ✓' if jobs_ok else '⚠️ non disponible (voir les logs)'))
                    print('\n💡 Connexion à AbiArt :')
                    print('   1. Sur votre site : Backend "Kaggle" → collez l\'URL ci-dessus.')
                    print(f'   2. Ou dans Railway : Variables → KAGGLE_URL = {url}')
                    if ABIART_APP_URL.strip():
                        try:
                            r = requests.post(f"{ABIART_APP_URL.rstrip('/')}/api/kaggle/register", json={'url': url}, timeout=8)
                            print('\n✅ Connecté automatiquement à AbiArt !' if r.status_code == 200
                                  else f'\nℹ️ Auto-enregistrement : HTTP {r.status_code}')
                        except Exception:
                            pass
                    print('\n(Laissez cette cellule tourner : l\'arrêter coupe le backend.)\n')
                else:
                    print('\n⚠️ Tunnel Cloudflare introuvable. Logs :\n' + open(TLOG).read()[-1500:])
        time.sleep(2)
except KeyboardInterrupt:
    print('\nArrêt demandé.')
finally:
    for p in (tunnel_proc, proc):
        try:
            if p and p.poll() is None:
                os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        except Exception:
            pass